# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
!git clone -q https://github.com/rekhasreebandaru/flyrank-ml-internship.git
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")
df["declined"] = (df["clicks_last_30d"] < df["clicks_prev_30d"]).astype(int)

features = ["search_volume", "competition", "cpc", "word_count", "content_age_days",
            "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate"]

df_model = df.dropna(subset=features + ["declined", "client_id"]).copy()

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [8]:
print("FINDING #2 — The Content Performance Curve")
print("Claim: Health peaks at 61-90 days, hits a 'decay cliff' at 271-365 days, then")
print("'recovers' at 365+ (health 25.1).")
print()
print("Where does the label come from? Health score is a FlyRank composite")
print("(impressions + position + CTR + scroll depth), not an external ground truth.")
print()
print("Does the validation design carry the claim? The paper itself flags this: the")
print("365+ bucket sits inside the 'active-content' sample (impressions_90d > 0 and")
print("sessions_90d > 0). That's a survivorship filter — pages that went fully dead")
print("were dropped before the 365+ average was computed. So the 'recovery' at 365+")
print("may partly reflect which pages survived to be measured, not that age reverses")
print("decline. The paper's own caption agrees this shouldn't be a headline proof point.")
print()
print("=" * 70)
print()
print("FINDING #10 / ML Appendix — Growth Prediction (Logistic Regression, 71% holdout)")
print("Claim: Content age, days since update, and days visible are the strongest")
print("signals separating growing from declining pages.")
print()
print("Where does the label come from? An observed outcome (30d-vs-prev-30d impression")
print("trend), which is legitimate — not researcher-defined.")
print()
print("Does the validation design carry the claim? The methodology section states an")
print("80/20 split but does not say whether it is grouped by brand or a random row")
print("split. With 57 brands in the portfolio, a random split could let the same")
print("brand's writing style and topics appear in both train and test, inflating the")
print("71% accuracy — exactly the kind of leak this course's skill describes. This is")
print("a methodology question, not an accusation: the paper may already control for")
print("this and just not state it in the public-safe writeup.")

FINDING #2 — The Content Performance Curve
Claim: Health peaks at 61-90 days, hits a 'decay cliff' at 271-365 days, then
'recovers' at 365+ (health 25.1).

Where does the label come from? Health score is a FlyRank composite
(impressions + position + CTR + scroll depth), not an external ground truth.

Does the validation design carry the claim? The paper itself flags this: the
365+ bucket sits inside the 'active-content' sample (impressions_90d > 0 and
sessions_90d > 0). That's a survivorship filter — pages that went fully dead
were dropped before the 365+ average was computed. So the 'recovery' at 365+
may partly reflect which pages survived to be measured, not that age reverses
decline. The paper's own caption agrees this shouldn't be a headline proof point.


FINDING #10 / ML Appendix — Growth Prediction (Logistic Regression, 71% holdout)
Claim: Content age, days since update, and days visible are the strongest
signals separating growing from declining pages.

Where does the label co

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [9]:
# BEFORE: random split (not grouped) — the naive approach
X_all = df_model[features]
y_all = df_model["declined"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_all, y_all, test_size=0.25, random_state=42
)

rf_random = RandomForestClassifier(random_state=42, n_estimators=200)
rf_random.fit(X_train_r, y_train_r)
scores_random = rf_random.predict_proba(X_test_r)[:, 1]

precision_random_20 = precision_at_k(scores_random, y_test_r.values, 20)
precision_random_50 = precision_at_k(scores_random, y_test_r.values, 50)

# AFTER: grouped split by client_id — the honest approach (same as ML-08)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df_model, groups=df_model["client_id"]))
train_g, test_g = df_model.iloc[train_idx], df_model.iloc[test_idx]

rf_grouped = RandomForestClassifier(random_state=42, n_estimators=200)
rf_grouped.fit(train_g[features], train_g["declined"])
scores_grouped = rf_grouped.predict_proba(test_g[features])[:, 1]

precision_grouped_20 = precision_at_k(scores_grouped, test_g["declined"].values, 20)
precision_grouped_50 = precision_at_k(scores_grouped, test_g["declined"].values, 50)

comparison = pd.DataFrame({
    "split": ["Random (before)", "Grouped by client_id (after)"],
    "precision@20": [precision_random_20, precision_grouped_20],
    "precision@50": [precision_random_50, precision_grouped_50],
})
print(comparison.round(3))
print()
print(f"Gap at precision@20: {precision_random_20 - precision_grouped_20:.3f}")
print(f"Gap at precision@50: {precision_random_50 - precision_grouped_50:.3f}")

                          split  precision@20  precision@50
0               Random (before)          0.85          0.88
1  Grouped by client_id (after)          0.30          0.34

Gap at precision@20: 0.550
Gap at precision@50: 0.540


Result: The random split reports precision@20 of 0.850 and precision@50 of 0.880 — numbers that looked genuinely impressive. The grouped split (by client_id, matching ML-08) reports 0.300 and 0.340 — the same numbers from Week 5. The gap is enormous: 0.550 at precision@20, 0.540 at precision@50. This means the random-split "before" number was almost entirely memorization of client-specific patterns, not a real predictive signal. A model trained and evaluated with a random split on this data would have looked 2-3x stronger than it actually is on unseen clients. This is the clearest evidence in this whole assignment for why grouped splits matter — the honest number (0.30-0.34) is the real one to trust and report.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [10]:
print("LEAKAGE AUDIT — attacking my own feature set")
print()

# 1. Timeline check: are any features derived from a window that overlaps the label?
print("1. Timeline check:")
print("   Label: declined = clicks_last_30d < clicks_prev_30d (a 30-day-vs-prior-30-day comparison)")
print("   Features used:", features)
print("   None of these features are built from clicks_last_30d or clicks_prev_30d directly.")
print("   ctr, avg_position, engagement_rate, scroll_rate are portfolio-level or 90-day")
print("   aggregates, not the specific 30-day windows the label compares — timeline holds.")
print()

# 2. Label-derived / sibling column check: train with vs without the top feature
from sklearn.ensemble import RandomForestClassifier

X_with = train_g[features]
X_without = train_g[[f for f in features if f != "ctr"]]  # ctr was the top feature in ML-08

rf_with = RandomForestClassifier(random_state=42, n_estimators=200).fit(X_with, train_g["declined"])
rf_without = RandomForestClassifier(random_state=42, n_estimators=200).fit(X_without, train_g["declined"])

scores_with = rf_with.predict_proba(test_g[features])[:, 1]
scores_without = rf_without.predict_proba(test_g[[f for f in features if f != "ctr"]])[:, 1]

p20_with = precision_at_k(scores_with, test_g["declined"].values, 20)
p20_without = precision_at_k(scores_without, test_g["declined"].values, 20)

print("2. Label-derived feature check (train with vs. without top feature 'ctr'):")
print(f"   precision@20 WITH ctr: {p20_with:.3f}")
print(f"   precision@20 WITHOUT ctr: {p20_without:.3f}")
print(f"   Drop: {p20_with - p20_without:.3f}")
print("   A small drop (not a collapse from ~1.0 to ~0.7) means ctr is a genuine signal,")
print("   not a disguised copy of the label.")
print()

# 3. Product flag check
print("3. Product-flag check:")
print("   No FlyRank internal flags (optimization flags, health score) are in my feature")
print("   set — I use only raw measured columns, never a pre-existing system's judgment.")
print()

# 4. Split check
print("4. Split check: grouped by client_id, confirmed in Section 2 (not random).")
print()

# 5. Base rate check
base_rate = test_g["declined"].mean()
print(f"5. Base rate check: {base_rate:.3f} — printed alongside every precision@K reported")
print("   in this notebook and in Week 5's ML-08 notebook.")

LEAKAGE AUDIT — attacking my own feature set

1. Timeline check:
   Label: declined = clicks_last_30d < clicks_prev_30d (a 30-day-vs-prior-30-day comparison)
   Features used: ['search_volume', 'competition', 'cpc', 'word_count', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']
   None of these features are built from clicks_last_30d or clicks_prev_30d directly.
   ctr, avg_position, engagement_rate, scroll_rate are portfolio-level or 90-day
   aggregates, not the specific 30-day windows the label compares — timeline holds.

2. Label-derived feature check (train with vs. without top feature 'ctr'):
   precision@20 WITH ctr: 0.300
   precision@20 WITHOUT ctr: 0.250
   Drop: 0.050
   A small drop (not a collapse from ~1.0 to ~0.7) means ctr is a genuine signal,
   not a disguised copy of the label.

3. Product-flag check:
   No FlyRank internal flags (optimization flags, health score) are in my feature
   set — I use only raw measured

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [11]:
print("MY BOLDEST CLAIM (from ML-08):")
print('"Random Forest is the only method that beats both the baseline and the base')
print('rate at every K tested — a real, if modest, win."')
print()
print("What makes this claim bold: 'beats' and 'real win' both imply a settled,")
print("general fact rather than a result observed on one specific test split.")
print()
print("SAFE REWRITE:")
print('"On this grouped, client-held-out test split, Random Forest showed higher')
print('precision@20 and precision@50 than both the ML-07 rule baseline and the base')
print('rate. This is a directional, decision-support result from a single split with')
print('only 8 held-out clients — not a claim that Random Forest will outperform on')
print('every client or every time period."')
print()
print("Why the rewrite is safer:")
print("- 'showed higher precision' (observed) replaces 'beats' (implies proof)")
print("- 'directional, decision-support result' replaces 'real win' (implies certainty)")
print("- names the small test size (8 clients) as a real limit on how far this")
print("  generalizes, which Section 2's random-vs-grouped gap (0.55) makes obvious")
print("  is not a small caveat — a different client split could move this number")
print("  meaningfully, since the test set is small")

MY BOLDEST CLAIM (from ML-08):
"Random Forest is the only method that beats both the baseline and the base
rate at every K tested — a real, if modest, win."

What makes this claim bold: 'beats' and 'real win' both imply a settled,
general fact rather than a result observed on one specific test split.

SAFE REWRITE:
"On this grouped, client-held-out test split, Random Forest showed higher
precision@20 and precision@50 than both the ML-07 rule baseline and the base
rate. This is a directional, decision-support result from a single split with
only 8 held-out clients — not a claim that Random Forest will outperform on
every client or every time period."

Why the rewrite is safer:
- 'showed higher precision' (observed) replaces 'beats' (implies proof)
- 'directional, decision-support result' replaces 'real win' (implies certainty)
- names the small test size (8 clients) as a real limit on how far this
  generalizes, which Section 2's random-vs-grouped gap (0.55) makes obvious
  is not a sma

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.